In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
import sklearn.pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
import dask.dataframe as dd
from sklearn.model_selection import RandomizedSearchCV


In [2]:
import joblib

In [3]:
rf_pipeline = joblib.load('rf_pipeline.joblib')

e:\OneDrive\Documents\coding\MLInternship\myenv\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
e:\OneDrive\Documents\coding\MLInternship\myenv\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
e:\OneDrive\Documents\coding\MLInternship\myenv\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator RandomFore

In [4]:
type_labelencoder = joblib.load('type_labelencoder.joblib')

e:\OneDrive\Documents\coding\MLInternship\myenv\Lib\site-packages\sklearn\base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
def safe_label_transform(le, values):
    known_classes = set(le.classes_)
    encoded = []
    for v in values:
        if v in known_classes:
            encoded.append(le.transform([v])[0])
        else:
            # Map unseen types to -1 (or max+1, or any other value you want)
            encoded.append(-1)
    return encoded

data['type_encoded'] = safe_label_transform(type_labelencoder, data['type'])

In [5]:
np.random.seed(42)
n_samples = 50  # You can increase for bigger test set

# Main (log-normal, skewed) distributions
amount = np.round(np.random.lognormal(mean=9, sigma=1, size=n_samples), 2)  # mean~8k, long tail
oldbalanceOrg = np.round(amount + np.abs(np.random.lognormal(mean=8, sigma=1, size=n_samples)), 2)
newbalanceOrig = oldbalanceOrg - amount

oldbalanceDest = np.round(np.abs(np.random.lognormal(mean=9, sigma=1, size=n_samples)), 2)
newbalanceDest = oldbalanceDest + amount

# Transaction types (add one rare/unseen type for test)
type_choices = ["TRANSFER", "CASH_OUT", "PAYMENT"]
type_probs = [0.47, 0.47, 0.06]
type_ = np.random.choice(type_choices, n_samples, p=type_probs)

step = np.random.randint(1, 30, n_samples)

errorBalanceOrig = oldbalanceOrg - amount - newbalanceOrig
errorBalanceDest = oldbalanceDest + amount - newbalanceDest
isOrigZero = (oldbalanceOrg == 0).astype(int)
isDestZero = (oldbalanceDest == 0).astype(int)
isOrigNeg = (newbalanceOrig < 0).astype(int)
isDestNeg = (newbalanceDest < 0).astype(int)
amt_perc_orig = np.divide(amount, oldbalanceOrg + 1)
amt_perc_dest = np.divide(amount, oldbalanceDest + 1)
anomaly_score = np.round(np.random.normal(0, 1, n_samples), 2)

data = pd.DataFrame({
    "step": step,
    "type": type_,
    "amount": amount,
    "oldbalanceOrg": oldbalanceOrg,
    "newbalanceOrig": newbalanceOrig,
    "oldbalanceDest": oldbalanceDest,
    "newbalanceDest": newbalanceDest,
    "errorBalanceOrig": errorBalanceOrig,
    "errorBalanceDest": errorBalanceDest,
    "isOrigZero": isOrigZero,
    "isDestZero": isDestZero,
    "isOrigNeg": isOrigNeg,
    "isDestNeg": isDestNeg,
    "amt_perc_orig": amt_perc_orig,
    "amt_perc_dest": amt_perc_dest,
    "anomaly_score": anomaly_score,
})

# --- Inject specific edge/rare cases at the end ---
edge_cases = [
    # All zeros
    {"step": 1, "type": "TRANSFER", "amount": 0, "oldbalanceOrg": 0, "newbalanceOrig": 0,
     "oldbalanceDest": 0, "newbalanceDest": 0, "errorBalanceOrig": 0, "errorBalanceDest": 0,
     "isOrigZero": 1, "isDestZero": 1, "isOrigNeg": 0, "isDestNeg": 0, "amt_perc_orig": 0, "amt_perc_dest": 0, "anomaly_score": 0},
    # Negative balances
    {"step": 5, "type": "CASH_OUT", "amount": 10000, "oldbalanceOrg": 5000, "newbalanceOrig": -5000,
     "oldbalanceDest": 30000, "newbalanceDest": 40000, "errorBalanceOrig": 0, "errorBalanceDest": 0,
     "isOrigZero": 0, "isDestZero": 0, "isOrigNeg": 1, "isDestNeg": 0, "amt_perc_orig": 2, "amt_perc_dest": 0.25, "anomaly_score": 3.5},
    # Amount equals old balance
    {"step": 3, "type": "TRANSFER", "amount": 20000, "oldbalanceOrg": 20000, "newbalanceOrig": 0,
     "oldbalanceDest": 1000, "newbalanceDest": 21000, "errorBalanceOrig": 0, "errorBalanceDest": 9000,
     "isOrigZero": 0, "isDestZero": 0, "isOrigNeg": 0, "isDestNeg": 0, "amt_perc_orig": 1, "amt_perc_dest": 20, "anomaly_score": 4.8},
    # Extremely high amount
    {"step": 7, "type": "PAYMENT", "amount": 5000000, "oldbalanceOrg": 10000, "newbalanceOrig": -4990000,
     "oldbalanceDest": 0, "newbalanceDest": 5000000, "errorBalanceOrig": 0, "errorBalanceDest": 0,
     "isOrigZero": 0, "isDestZero": 1, "isOrigNeg": 1, "isDestNeg": 0, "amt_perc_orig": 500, "amt_perc_dest": 5000, "anomaly_score": 5.7},
    # Unknown type
    {"step": 15, "type": "REVERSAL", "amount": 8000, "oldbalanceOrg": 8000, "newbalanceOrig": 0,
     "oldbalanceDest": 20000, "newbalanceDest": 28000, "errorBalanceOrig": 0, "errorBalanceDest": 0,
     "isOrigZero": 0, "isDestZero": 0, "isOrigNeg": 0, "isDestNeg": 0, "amt_perc_orig": 1, "amt_perc_dest": 0.4, "anomaly_score": 0.6}
]
data = pd.concat([data, pd.DataFrame(edge_cases)], ignore_index=True)

# Now you have a DataFrame with realistic + edge case transactions!
print(data.tail(10))
# You can now use `data` in your model prediction demo, SHAP analysis, etc.


    step      type      amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
45    26  TRANSFER     3944.81        4634.67          689.86        17708.90   
46     7  TRANSFER     5112.08        9120.37         4008.29         2352.07   
47    25  CASH_OUT    23321.30       27191.48         3870.18         2163.63   
48     4  TRANSFER    11425.67       14421.91         2996.24        13656.10   
49    13   PAYMENT     1389.86        3747.49         2357.63        10905.09   
50     1  TRANSFER        0.00           0.00            0.00            0.00   
51     5  CASH_OUT    10000.00        5000.00        -5000.00        30000.00   
52     3  TRANSFER    20000.00       20000.00            0.00         1000.00   
53     7   PAYMENT  5000000.00       10000.00     -4990000.00            0.00   
54    15  REVERSAL     8000.00        8000.00            0.00        20000.00   

    newbalanceDest  errorBalanceOrig  errorBalanceDest  isOrigZero  \
45        21653.71               0.0  

In [15]:
print (data.head())

   step      type    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0     3  CASH_OUT  13315.90       17437.86         4121.96         1967.72   
1    28  CASH_OUT   7056.72        9084.95         2028.23         5320.67   
2     1  CASH_OUT  15485.95       17000.81         1514.86         5751.90   
3    20  TRANSFER  37161.55       42657.00         5495.45         3632.67   
4    29  TRANSFER   6411.49       14769.70         8358.21         6896.12   

   newbalanceDest  errorBalanceOrig  errorBalanceDest  isOrigZero  isDestZero  \
0        15283.62               0.0               0.0           0           0   
1        12377.39               0.0               0.0           0           0   
2        21237.85               0.0               0.0           0           0   
3        40794.22               0.0               0.0           0           0   
4        13307.61               0.0               0.0           0           0   

   isOrigNeg  isDestNeg  amt_perc_orig  amt_

In [7]:
# This gets the features the pipeline expects (from training time)
expected_features = rf_pipeline.named_steps['scaler'].feature_names_in_


In [8]:
# If you used 'type_encoded' in training, replace 'type' with 'type_encoded'
X_demo = data.copy()
if 'type' in expected_features and 'type_encoded' in X_demo.columns:
    X_demo['type'] = X_demo['type_encoded']

X_demo = X_demo[expected_features]  # Keep exact features/order


In [9]:
y_pred = rf_pipeline.predict(X_demo)


In [ ]:
print (y_pred)

In [10]:
import shap

# Extract the trained Random Forest model from the pipeline
model = rf_pipeline.named_steps['clf']

# Create a SHAP explainer object
# For tree-based models, TreeExplainer is recommended
explainer = shap.TreeExplainer(model)

e:\OneDrive\Documents\coding\MLInternship\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
print(rf_pipeline.named_steps)

{'scaler': StandardScaler(), 'clf': RandomForestClassifier(random_state=42)}


In [12]:
print("Columns in SHAP input:", list(X_demo.columns))
print("Shape of SHAP input:", X_demo.shape)

Columns in SHAP input: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'isOrigZero', 'isDestZero', 'isOrigNeg', 'isDestNeg', 'amt_perc_orig', 'amt_perc_dest']
Shape of SHAP input: (55, 15)


In [13]:
for name, imp in zip(X_demo, model.feature_importances_):
    print(f"{name:15s}: {imp:.4f}")

step           : 0.0161
type           : 0.0207
amount         : 0.0702
oldbalanceOrg  : 0.0324
newbalanceOrig : 0.4734
oldbalanceDest : 0.0248
newbalanceDest : 0.0490
errorBalanceOrig: 0.1131
errorBalanceDest: 0.0435
isOrigZero     : 0.0026
isDestZero     : 0.0134
isOrigNeg      : 0.0000
isDestNeg      : 0.0000
amt_perc_orig  : 0.0849
amt_perc_dest  : 0.0560


In [14]:
# Calculate SHAP values
shap_values = explainer.shap_values(X_demo)

In [ ]:
# Visualize the mean absolute SHAP values (bar plot)
shap.summary_plot(shap_values, X_demo, plot_type="bar")

In [15]:
# Assuming y_pred contains the model predictions (0 for not fraud, 1 for fraud)
fraud_indices = data[y_pred == 1].index

print(f"Model predicted fraud for {len(fraud_indices)} instances.")

# Iterate through the fraudulent instances and explain each one
for idx in fraud_indices:
    print(f"\nExplaining instance at index {idx}:")
    # Get the data for this instance
    instance_data = X_demo.loc[[idx]]

    # Calculate SHAP values for this specific instance
    instance_shap_values = explainer.shap_values(instance_data)

    # Print explanation in a more readable format
    print("This instance was flagged as fraud due to the following reasons:")
    # Get SHAP values for the fraud class (index 1) for this instance
    shap_values_fraud_class = instance_shap_values[0, :, 1]

    # Sort features by their absolute SHAP value to show most influential first
    sorted_indices = abs(shap_values_fraud_class).argsort()[::-1]

    for i in sorted_indices:
       feature = X_demo.columns[i]
       shap_value = shap_values_fraud_class[i]
       # You can add a condition here to only print features with significant SHAP values
       # For simplicity, printing all here, but you could add e.g., if abs(shap_value) > threshold:
       print(f"  - {feature}: {shap_value:.4f}")


Model predicted fraud for 4 instances.

Explaining instance at index 51:
This instance was flagged as fraud due to the following reasons:
  - amount: 0.3169
  - newbalanceOrig: 0.0598
  - errorBalanceOrig: -0.0442
  - amt_perc_orig: -0.0417
  - oldbalanceOrg: 0.0224
  - newbalanceDest: -0.0062
  - errorBalanceDest: 0.0039
  - isOrigZero: 0.0035
  - type: 0.0030
  - amt_perc_dest: 0.0023
  - step: -0.0017
  - oldbalanceDest: 0.0009
  - isDestZero: 0.0006
  - isOrigNeg: 0.0000
  - isDestNeg: 0.0000

Explaining instance at index 52:
This instance was flagged as fraud due to the following reasons:
  - amount: 0.2857
  - errorBalanceDest: 0.0559
  - amt_perc_dest: 0.0326
  - oldbalanceOrg: 0.0303
  - amt_perc_orig: -0.0237
  - errorBalanceOrig: -0.0203
  - oldbalanceDest: -0.0168
  - isDestZero: -0.0147
  - type: -0.0066
  - newbalanceOrig: -0.0063
  - newbalanceDest: -0.0062
  - isOrigZero: -0.0011
  - step: 0.0006
  - isOrigNeg: 0.0000
  - isDestNeg: 0.0000

Explaining instance at index 5

In [47]:
def plain_english_reason(feature, value):
    # Add plain English mappings for your features
    if feature == "isOrigNeg" and value == 1:
        return "The sender's balance became negative after the transaction."
    elif feature == "isOrigZero" and value == 1:
        return "The sender's account started with a zero balance."
    elif feature == "isDestNeg" and value == 1:
        return "The receiver's balance became negative."
    elif feature == "isDestZero" and value == 1:
        return "The receiver's account started with a zero balance."
    elif feature == "amt_perc_orig" and value > 0.9:
        return "Almost all the sender's balance was transferred."
    elif feature == "amount" and value > 1_000_000:
        return "The transaction amount is unusually high."
    elif feature == "errorBalanceOrig" and abs(value) > 1e-3:
        return "There is a mismatch in the sender's balance after the transaction."
    elif feature == "errorBalanceDest" and abs(value) > 1e-3:
        return "There is a mismatch in the receiver's balance after the transaction."
    elif feature == "type":
        # Adjust as needed for your encoding
        return f"The transaction type is: {value}"
    elif feature == "step":
        return f"This transaction occurred at step {value} in the simulation."
    else:
        return f"{feature.replace('_',' ').capitalize()} has a value of {value}."

# For each fraud instance, print plain-English reasons:
N = 3  # Number of top reasons to show

for idx in fraud_indices:
    print(f"\nTransaction at index {idx} was flagged as fraud for these reasons:")

    instance_data = X_demo.loc[[idx]]
    instance_shap_values = explainer.shap_values(instance_data)
    shap_values_fraud_class = instance_shap_values[0, :, 1]

    sorted_indices = abs(shap_values_fraud_class).argsort()[::-1]
    # Only take top N
    for rank, i in enumerate(sorted_indices[:N]):
        feature = X_demo.columns[i]
        value = instance_data.iloc[0][feature]
        reason = plain_english_reason(feature, value)
        if rank == 0:
            print(f" - Most important reason: {reason}")
        else:
            print(f" - Also because: {reason}")



Transaction at index 51 was flagged as fraud for these reasons:
 - Most important reason: Amount has a value of 10000.0.
 - Also because: Newbalanceorig has a value of -5000.0.
 - Also because: Errorbalanceorig has a value of 0.0.

Transaction at index 52 was flagged as fraud for these reasons:
 - Most important reason: Amount has a value of 20000.0.
 - Also because: There is a mismatch in the receiver's balance after the transaction.
 - Also because: Amt perc dest has a value of 20.0.

Transaction at index 53 was flagged as fraud for these reasons:
 - Most important reason: The transaction amount is unusually high.
 - Also because: Amt perc dest has a value of 5000.0.
 - Also because: Almost all the sender's balance was transferred.

Transaction at index 54 was flagged as fraud for these reasons:
 - Most important reason: Amount has a value of 8000.0.
 - Also because: Almost all the sender's balance was transferred.
 - Also because: Oldbalanceorg has a value of 8000.0.


In [16]:
import joblib
joblib.dump (explainer, 'shap_explainer.joblib')




['shap_explainer.joblib']